# Lab 3 — Getting the time axis right
### EP0418 · Topic 3 · 2 practical hours

**You need:** nothing to download: the notebook fetches `ep0418.py` and
`sg_taxi_hourly.csv` itself. Have the printed Lab 3 sheet beside you for the boxes
you write in.

---
Singapore's taxi feed on `data.gov.sg` reports how many taxis are free across the
island, about once an hour. It records the moment each reading was taken, so the
readings are about an hour apart, not exactly. Before you can compare one hour with
another, you have to get the time axis right. The lab does it in five steps, one task
each. Each task opens with this picture again, its own step outlined, so you always
know where you are.

![The plan in five steps: Task 1 read the times, Task 2 put them on a grid, Task 3 count the empty hours, Task 4 same hour last week, Task 5 see the cycles](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_plan.png)

Today you:

- read the timestamps and see how they drift (Task 1);
- put the readings on a regular hourly grid (Task 2) and count the empty hours (Task 3);
- compare each hour with the same hour last week (Task 4);
- see the daily and weekly cycles (Task 5).

## How the cells work

- **Grey code cells:** run each one when you reach it, top to bottom. Click in it
  and press Shift+Enter.
- **Text boxes** have a bold label ending in a colon, like **My answers:**.
  Double-click, type, press Shift+Enter. Or write on the printed sheet.
- **Gemini's code** arrives as a **green cell**. Click **Accept and run** (bottom of
  the chat panel), then run the task's check cell.
- **Check cells** start with `# TODO(you):`, which says what to run first. Never
  change them: if Gemini shows red and green lines inside one, click **Cancel**
  (red and green lines inside Gemini's own green cell are fine). Leave lines like
  `s = keep("s")` as they are. If a check cell says *Not there yet* or *No ... yet*,
  type in the chat *call the result* and the name the task asks for, accept, and
  run the check cell again.
- **Gemini's lab rules** (*EP0418 lab rules*) are built in. If a box *Review custom
  instructions* appears, click **Use custom instruction**. The sliders icon beside
  **+** in the chat box should be blue; if it is grey, click it and choose *EP0418
  lab rules*. Keep *Learn mode* off.
- **If Gemini says *Daily limit reached*:** paste the same prompt into
  ChatGPT, Claude or Copilot in a new tab. It cannot see your notebook: copy its code
  into a new cell (**+ Code** at the top of the page) and run it, and paste any
  failed check or red error back into its chat. Do the quiz last to save your limit.
- **Prompts** sit in grey quote boxes. When you change one, keep its rules (for
  example, do not drop rows; show the row count before and after): they matter as much
  as the task. A prompt with no example is **zero-shot**; one example of the output
  you want makes it **one-shot**, several make it **few-shot**.
- **Checks** (the `check.*` lines under a task) are the **verification contract**.
  Do not edit them. In industry they are called **evals**.
- **A failing check** says what it found and what to try. Failing checks are normal
  while you work.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. If a box *Review custom instructions* appears while you save the copy, click **Make a copy**: it keeps the lab's short rules for the AI assistant. Colab opens your copy in a new tab with *Copy of* at the start of its name at the top left: work in that tab and close the old one. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    global _NAMED
    v = globals().get(name, ...)
    if v is ... or v is None:
        print(f"Not there yet: nothing called {name}. Not run Gemini's green cell yet? Click Accept and "
              f"run on it, then run this cell again. Already ran it? Type in the same chat: call the "
              f"result {name}. Then Accept and run, and run this cell again.")
        _NAMED = True
        return ...
    return v


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    global _NAMED
    if any(v is ... or v is None for v in values):
        if not globals().get("_NAMED"):      # keep() has already said exactly what is missing
            print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
                  "or paste its code into a new cell and run it), then run this cell again.")
        _NAMED = False
        return False
    _NAMED = False
    return True


def show_predict(data, unit):
    """Prediction League: run the newest predict that Gemini wrote and print its code to copy."""
    ran = globals().get("In", [])                                   # every cell you have run, oldest first
    mine = [c for c in ran if "def predict(" in c and "The baseline predicts" not in c and "show_predict" not in c]
    if not mine:
        print("Gemini's predict has not run yet. Find the cell Gemini added in Step 2 (it starts with")
        print("def predict, or with import lines just above it), click in it, press Shift+Enter,")
        print("then run this cell again."); return
    code = mine[-1].strip()
    for short, line in (("np.", "import numpy as np"), ("pd.", "import pandas as pd")):
        if short in code and line not in code:                     # the league page starts empty
            code = line + "\n" + code
    ns = {}
    exec(code, ns)
    print("Your predict says", round(ns["predict"](data.copy()), 1), unit)
    print()
    print("----- copy from the line below -----")
    print(code)
    print("----- to the line above -----")

CSV = "sg_taxi_hourly.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
print(f"{len(raw):,} rows")
raw.head()

## Task 1 — Read the timestamps before anything else (20 minutes)

![Where you are: step 1 of 5, read the times](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_plan_step1.png)

*Topic 3: the time column is data too. Readings are not always taken exactly on the hour.*

**What each field means.** The file `sg_taxi_hourly.csv`, loaded as `raw` by the cell
above, has 1,344 readings from 29 May to 24 July 2026: eight weeks. Every time ends in
`+08:00`. That means Singapore time, which is 8 hours ahead of UTC, the world's
reference clock. The time shown, such as `23:59:48`, is already Singapore time: do not
add 8 hours to it.

| Where | Field | What it means | Example |
|---|---|---|---|
| `raw` | `timestamp` | the moment the reading was taken, to the second; `+08:00` means Singapore time | 2026-05-29 23:59:48+08:00 |
| `raw` | `taxi_count` | how many taxis were free for hire across Singapore at that moment | 3008 |

Task 1 has two parts: **1A**, read the first six timestamps; **1B**, chart the
readings and the gaps between them.

### 1A · Read the first six timestamps

1. **Run the cell below.** It prints the first six timestamps, then the five most
   common gaps between one reading and the next.
2. **Read the times.** Are they exactly on the hour, like `01:00:00`? Write your
   answer in the box *On the hour?* just under the code cell, before reading on.

In [ ]:
print(raw.timestamp.head(6).to_string(index=False))
print()
print("gaps between consecutive readings:")
print(raw.timestamp.diff().value_counts().head(5).to_string())

**On the hour? (yes / no, and an example time):**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

No. The first reading is `23:59:48`, then `00:59:50`, `01:59:53`, `02:59:55`: each
one a few seconds before the hour, and creeping a couple of seconds later every
time. A good answer gives one of those times as the example. The common wrong answer
is "yes", either because the times look close enough or because the sixth reading
happens to be `05:00:00`. That one is on the hour by luck, not by design. The feed
records the moment the reading was taken, so the gaps are about one hour, not
exactly one hour, and that is why the readings must go on a regular grid before you
count back 168 hours to the same hour last week (Task 4).
</details>

**What you should see:** times like `23:59:48` and `00:59:50`, a few seconds off
the hour. The five most common gaps are `01:00:02` (423 times), `01:00:03` (392),
`00:59:33` (141), `00:59:32` (129) and `01:00:33` (111): all about one hour, none
exactly one hour.

### 1B · Chart the readings and the gaps between them

1. **Run the cell below.** It draws two charts, every reading and the time since
   the previous reading, then prints two lines.
2. **Look at the second chart.** Each point is one reading. Its height is the time
   since the reading before it, in hours: 1.0 is exactly one hour, 1.2 is
   72 minutes, 0.6 is 36 minutes.

In [ ]:
show.series(raw, col="taxi_count", title="Taxis available, every reading, eight weeks")
hours_since_previous = raw.timestamp.diff().dt.total_seconds() / 3600
show.series(pd.Series(hours_since_previous.values, index=raw.timestamp),
            title="Hours since the previous reading", ylabel="hours", hline=1)
off = (hours_since_previous - 1).abs() > 2 / 60          # more than 2 minutes from 1 hour
print(f"readings more than 2 minutes early or late: {int(off.sum())} of {len(raw):,}")
print(f"shortest gap: {hours_since_previous.min() * 60:.0f} min   longest gap: {hours_since_previous.max() * 60:.0f} min")

**What you should see**, chart by chart:

- **Chart 1, the taxi counts.** Every reading over eight weeks. It goes up and down a
  lot; Task 5 sorts out the daily and weekly pattern. Two stretches sit far below the
  rest, on 13 June and from 17 to 19 July: leave them for now, Lab 4 is about values
  like these. Nothing in this chart shows a problem with the times.
- **Chart 2, the time between readings.** Almost every point sits on the dashed
  1-hour line. The few spikes are readings that came early or late.
- **The two printed lines** count them: `readings more than 2 minutes early or late:
  21 of 1,344`, then `shortest gap: 34 min   longest gap: 71 min`.

**The main point**, in four lines:

- The readings are about an hour apart, so both charts look fine.
- But each reading comes a few seconds later than the one before. Now and then one
  slips past the hour mark: `03:59:58`, then `05:00:00`.
- So the 04:00 hour has no reading at all, and chart 2 cannot show it: the gap is still
  a normal hour.
- Task 2 gives every clock hour its own slot, so the empty hours show up. The clip
  below shows it on six real readings.

**What neither chart can show**, on six real readings from 30 May (it plays slowly, about a minute):
![Six real readings from 30 May drift across the hour mark: the 04:00 hour has no reading and the 05:00 hour has two, although the gap between them is a normal 1 h 0 min 2 s](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_drift.gif)

### Why it matters: a shift counts rows, not hours

To compare each hour with the same hour last week, you count back one week (168 hours)
with `.shift(168)`. But `shift` counts **rows**, not hours: `shift(168)` means "the
value 168 rows earlier".

- **In this file, counting rows is close enough.** The feed never missed a reading, so
  168 rows back always lands within half an hour of one week back.
- **That only works while no reading goes missing.** Real feeds do miss readings. If
  this one had lost 3 readings early on Wednesday 10 June, then for the whole next week
  every "168 rows back" would land 3 hours too early. Thursday 11 June 08:59 would be
  compared with Thursday 4 June 05:59, the quiet early morning, instead of 08:59. No
  error would appear.
- **A regular grid fixes this.** Every clock hour gets its own slot, and a lost reading
  leaves an empty slot instead of disappearing. One slot is always one hour, so
  `shift(168)` is exactly one week back, whatever goes missing.

**Watch**, six real readings from 30 May, then the same six with one reading lost (it plays slowly, about a minute and a half):
![shift(2) on six real rows goes back about 2 hours; after one reading is lost the rows close up and shift(2) goes back 3 hours with no warning; on the hourly grid the lost reading is an empty slot and 2 slots back is still 2 hours](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_rows_slots.gif)

## Task 2 — Put the readings on a regular hourly grid (30 minutes)

![Where you are: step 2 of 5, put them on a grid](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_plan_step2.png)

*Topic 3, Resampling: you choose the slot size, what goes in each slot, and what an
empty slot shows.*

**Why a grid?** Task 4 compares each hour with the same hour last week. On a regular
grid:

- every clock hour has exactly one slot, so 168 slots back is always one week, even
  when readings go missing;
- an hour with no reading shows up as an empty slot, so you can count those hours
  (Task 3).

**How a reading gets its slot.** A reading goes in the slot of the clock hour it was
taken in. This is the rule `resample` uses, and the one this lab keeps:

- 05:00:00 and 05:59:33 were both taken in the 05:00 hour, so both go in the 05:00 slot.
- 03:59:58 was taken in the 03:00 hour, so it goes in the 03:00 slot, although it is
  nearly 4 o'clock.
- Most readings (1,305 of the 1,344) come just before the hour, at about xx:59. So the
  03:00 slot usually holds a reading taken just before 4 am.

In a real system the readings keep arriving and the grid grows hour by hour. Here all
eight weeks come from the saved file, so everyone builds the same grid and the checks
give everyone the same answer.

**The goal:** `s`, a Series with one slot per hour, every hour of the eight weeks,
from 29 May 23:00 to 24 July 22:00. The slot size is one hour; the other two
decisions are yours.

**Step 1. Answer two questions before you ask the AI.** On the grid every clock hour
gets one box, and each box holds one number. Gemini has to be told what goes in each
box: it cannot guess what you want. Here are two real hours from early on 30 May. The
05:00 hour caught two readings: 2,232 taxis at 05:00:00 and 1,766 taxis at 05:59:33.
The 04:00 hour caught no reading at all: the reading before came at 03:59:58, the one
after at 05:00:00. Write two short answers in the box *My decisions* below:

1. What one number should the 05:00 box hold? (for example the first reading, the
   last one, or their average: pick one and say why)
2. What should the 04:00 box hold: 0, the 03:00 number copied in, or nothing (left
   empty)? Say why. (What would *0 taxis* mean on a chart of free taxis?)

**My decisions:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 2. Try a one-line prompt, run it, and see what Gemini had to guess.** Open
Gemini (the **spark icon** at the bottom centre of the Colab window) and start a new
chat (click **⋮** at the top right of the Gemini panel, then **New chat**, then
**Yes**). Type this prompt into the chat and press Enter:

> **Prompt:** Put raw on an hourly grid.

When its green cell appears, click **Accept and run**. Then run the cell below. It
finds what Gemini's cell made and prints its name, its slots and its empty slots. It
also reloads `raw` from the file, in case Gemini's cell changed it.

In [ ]:
# TODO(you): Accept and run Gemini's green cell for the one-line prompt first, then run this cell.
_found = False
for _name, _v in list(globals().items()):
    _grid = isinstance(_v, (pd.Series, pd.DataFrame)) and isinstance(_v.index, pd.DatetimeIndex)
    if not _name.startswith("_") and _grid and (_v.index == _v.index.floor("h")).all():
        _empty = _v.isna().sum() if isinstance(_v, pd.Series) else _v.isna().any(axis=1).sum()
        print(f"Gemini's result: {_name}   slots: {len(_v):,}   empty: {int(_empty)}")
        _found = True
if not _found:
    print("No result from Gemini found yet: run its green cell first.")
if "timestamp" not in raw.columns or len(raw) != 1344:
    print("Gemini's cell changed raw itself, so raw was reloaded from the file.")
raw = pd.read_csv(CSV, parse_dates=["timestamp"])

**Write three short answers** in the box below:

1. What name did the result get? (the check later needs the name `s`)
2. How many slots, and how many empty? (the right grid has 1,344 slots, 39 of them
   empty)
3. Did the answer in the chat say how you could check that the result is right? (yes or no)

Each answer that differs is something the AI had to guess. If all three match, it
guessed well this time; a clear prompt makes it right every time. That is the rule for
any prompt: what you leave out, the AI guesses.

**What the one-line prompt left to Gemini:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 3. Write your own prompt, in four parts.** Each part answers one question. The
questions work for any prompt that asks for code, in this lab or any other. Write one
line for each part in the box *My prompt* below. You do not send it: in Step 4 you
compare it with ours.

1. **What I have:** what would someone who has never seen your notebook need to know
   about the data? Ask yourself: if a stranger thought the readings came exactly on the
   hour, what would they not bother to do? (here: `raw`, its two columns, and that the
   readings are about an hour apart, not exactly on the hour)
2. **What I want:** what should the result look like, and what is it called? Ask
   yourself: should one value come back for each reading, or for each clock hour? (here:
   one value for every clock hour, called `s`)
3. **How I will check:** what printed number would show it worked? Ask yourself: Step 2
   told you the right grid has 1,344 slots, 39 of them empty. What could the code print
   so you can compare? (here: the rows before and after, and how many hours are empty)
4. **My decisions:** where could two people do this differently? Say your choice, with
   one worked example. Ask yourself: which of the two hours from Step 1 would a stranger
   fill differently from you? (here: your two answers from Step 1, and the 05:00 hour on
   30 May)

**My prompt:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 4. Compare with ours, then send ours.** Start a new chat again (**⋮** › **New
chat** › **Yes**), so the one-line prompt does not mix in. Open our prompt below and
check it against yours, part by part.

<details><summary><b>Show the prompt</b> (ours: open it after you have written yours, then copy all of it into the chat and press Enter)</summary>

> **What I have:** a DataFrame `raw` with a `timestamp` column and a `taxi_count`
> column. The readings are about an hour apart, not exactly on the hour.
>
> **What I want:** resample onto a regular hourly grid, one value for every clock
> hour, and call the result `s` (a Series of taxi_count).
>
> **How I will check:** show the row count before and after, and how many hours are
> empty. Explain in plain words what the code does.
>
> **My decisions:** take the mean within each hour. One example: on 30 May the
> readings at 05:00:00 (2232 taxis) and 05:59:33 (1766 taxis) both fall in the 05:00
> hour, so that slot is 1999.0. An hour with no reading must keep its slot in s, left
> empty (NaN), not 0 and not copied from the hour before.

**Compare.** Did yours have all four parts? The part people most often leave out is
the last sentence: what an empty hour must show. It says two things:

- **keep its slot**: otherwise the empty hours vanish, `s` has fewer slots than
  there are hours, and a shift of 168 slots is no longer one week;
- **left empty, not 0**: an hour with no reading is not an hour with no taxis. Zeros
  would look like real readings and pull every average down.

Without that sentence Gemini has to guess, and a common guess is to fill the empty
hours. Send ours, so everyone's check gives the same answer.
</details>

**Why our prompt has a worked example.** It shows Gemini one finished box: on 30 May
the readings at 05:00:00 and 05:59:33 both fall in the 05:00 hour, so that box is
1,999.0. A prompt with one worked example is called **one-shot**; with no example it is
*zero-shot*, and with two or more, *few-shot*. Gemini copies the example's rule, so the
example must be right: a wrong example gets copied too. The example also answers
question 1 of Step 1: two readings in one hour give their average, 1,999.

**Step 5. Run Gemini's code, then the check.**

1. Gemini answers by adding a new code cell to your notebook, with a green edge. At
   the bottom of the chat panel, just above the box where you type, is a button
   **Accept and run**. Click it: this keeps the code and
   runs it.
2. If the code appears only in the chat and no green cell appears: copy the code,
   click **+ Code** at the top of the page, paste it into the new cell and run it.
3. If Gemini shows red and green lines inside the check cell (its first line starts
   `# TODO(you)`), click **Cancel**: check cells are never changed.
4. Colab scrolls to the green cell. Scroll back down to this task and run the check
   cell just below this text. You should see `[PASS] regular_cadence`.

**Step 6. Only if the check fails, or Gemini's green cell shows red error text:** in
the same chat, type one sentence with your numbers. For example, if `s` has 1,305 rows:

> **Prompt:** s has 1,305 rows, but the eight weeks have 1,344 hours. Keep every
> hour, with taxi_count left as NaN when it has no reading.

Click **Accept and run**, then run the check cell again.
Give the numbers, not just *it didn't work*: with the numbers, Gemini knows what to fix.

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `s`), then run this cell.
s = keep("s")
if filled(s):
    print("first slot:", s.index[0])

# ---- verification contract — do not edit ----
check.regular_cadence(s, "1h")
check.report()

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** `first slot: 2026-05-29 23:00:00+08:00`. If yours ends in `+00:00`, the times were moved to UTC: in the same chat, type the prompt *Keep the timestamps in Singapore time (+08:00), as they are in raw.*
- **The report:** `[PASS] regular_cadence(1h)`.
- **Gemini's green cell** printed the size of `s`: 1,344 slots, one per hour. Fewer means the empty hours were dropped.

**What the code had to do**, on six real readings from 30 May, and the three ways it can turn out (it plays slowly, about a minute):
![Resampling six real readings from 30 May into hourly slots: each slot takes the mean of its readings and the empty 04:00 slot stays NaN; then the three outcomes, A right, B dropped, C filled](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_grid.gif)

## Task 3 — Count the empty hours (10 minutes)

![Where you are: step 3 of 5, count the empty hours](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_plan_step3.png)

Resampling does not invent data. It makes a slot for every hour, so you can count
the hours that had no reading.

1. **Run the cell below.** It prints one line: the empty hours in `s`, out of all
   its slots.
2. **Compare** your line with the *Expected output* box and the table **A / B / C**,
   both just under the code cell.
3. **Read** the part *Why are they empty?* under the table, and run its cell.
4. **Write three answers** in the box *Empty hours, my outcome, and why*, or on the
   sheet. The three questions are above the box.

In [ ]:
if filled(s):
    print(f"empty hours: {int(s.isna().sum())} of {len(s):,} ({s.isna().mean():.1%})")

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** `empty hours: 39 of 1,344 (2.9%)`

**Read the line** and compare:

| | empty hours | slots in `s` | what it means |
|---|---|---|---|
| **A** | 39 | 1,344 | **Right.** Every hour has a slot; the 39 with no reading are empty. |
| **B** | 0 | 1,305 | **Wrong:** the empty hours were dropped. `regular_cadence` in Task 2 fails. |
| **C** | 0 | 1,344 | **Wrong, and no check catches it:** the empty hours were filled, with 0 or with the hour before. |

**If you got A:** go on to *Why are they empty?* below.

**If you got B or C:** ask Gemini to fix it. In the same chat as Task 2, type one
sentence with your numbers. For **B**:

> **Prompt:** s has 1,305 rows, but the eight weeks have 1,344 hours. Keep every
> hour, with taxi_count left as NaN when it has no reading. Keep everything else.

For **C**:

> **Prompt:** s has 1,344 rows but 0 empty hours. The 39 hours with no reading must
> stay NaN, not 0 and not copied from the hour before. Fix that and keep everything
> else.

Click **Accept and run**, run the Task 2 check cell, then run this task's count cell
again: it should now say 39 of 1,344. Give the numbers, not just *it didn't work*:
with the numbers, Gemini knows what to fix; without them, it guesses.

**Why are they empty?** An empty slot means no reading was **taken** in that clock hour. It does not mean a
reading was **lost**. **Run the cell below.** It counts the readings in each slot
around the first empty one, 04:00 on 30 May, then counts the slots that hold two.

In [ ]:
readings_per_slot = raw.set_index("timestamp").taxi_count.resample("h").count()
print(readings_per_slot["2026-05-30 02:00":"2026-05-30 06:00"].to_string())
print("slots with two readings:", int((readings_per_slot == 2).sum()))

**Expected output** *(computed from the data file when this notebook was built)*:

- **The five slots** from 02:00 to 06:00 hold 1, 1, **0**, **2** and 1 readings.
- **Last line:** `slots with two readings: 39`

**What the cell shows**, in three lines:

- The 04:00 slot is empty and the 05:00 slot holds two readings. The second one,
  taken at 05:59:33, is really the 6 o'clock reading, a little early. Nothing was lost.
- 39 slots hold two readings, the same as the 39 empty slots. Every empty slot is
  followed by a slot with two readings.
- Another rule, rounding each reading to the nearest hour, would put 05:59:33 in the
  06:00 slot and leave no slot empty. This lab keeps the clock-hour rule, the one
  `resample` uses, so everyone gets the same 39.

**The lesson:** before you call an empty slot lost data, look at the slots next to it.

**Write down three answers** in the box below:

1. How many empty hours, out of how many slots? (your line, `empty hours: … of …`)
2. Which outcome, A, B or C, matches your line? (the table **A / B / C**)
3. Was a reading lost in the 04:00 hour on 30 May? Say how you know.

**Empty hours, my outcome, and why:**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Yours should read 39 of 1,344 (2.9%): outcome A. If yours says
0 of 1,344 (outcome C), Gemini filled the empty hours with something, usually zeros
or the previous value, and the chart in Task 2 dropped to 0 or ran flat in places.
If yours says 0 of 1,305 (outcome B), the empty hours were dropped instead (1,344
minus 39). Both look tidy and both are wrong, because an hour with no reading is not
an hour with no taxis. For question 3: no reading was lost. The 05:00 slot holds two
readings: the 6 o'clock reading came early, at 05:59:33, so the 04:00 slot got none.
</details>

## Task 4 — Compare each hour with the same hour last week (20 minutes)

![Where you are: step 4 of 5, same hour last week](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_plan_step4.png)

**What we are trying to do.** Set the score to beat. Before anyone builds a clever forecast
(Lab 6), we need to know how wrong the laziest possible guess is. A forecast that cannot beat
the lazy guess is not worth using.

**What is the guess?** The zero-cleverness guess: whatever happened at that same hour seven
days ago. Wednesday 6 pm this week? Look up Wednesday 6 pm last week and say that number.
No maths, just a copy. This is called *seasonal naive*: *naive* because it only copies,
*seasonal* because it copies from one cycle back, and the cycle here is a week. On the hourly
grid a week is 168 slots, so `shift(168)` makes the copy for every hour at once.

**Why a week back, not a day back?** Taxis run on a weekly rhythm: a Sunday looks nothing
like a Monday, but last Monday looks a lot like this Monday.

- Weekends are quieter: about 1,880 taxis free on an average Saturday hour and 1,940 on a
  Sunday, against 2,400 to 2,650 on weekdays.
- Guessing a Monday hour from the Sunday before misses by about 736 taxis; from the Monday
  before, by about 389.
- Over every hour, a week back misses by about 496 taxis and a day back by about 532.

**What is MAE?** Then measure how wrong the copy was. For every hour, compare the copied
number with what really happened. Some guesses are too high, some too low. Ignore the
direction, keep the size of each miss, and average those sizes. That average is the MAE
(mean absolute error). For taxis it comes out around 496, so the dumb copy is typically
about 500 taxis off. The three steps, on three real hours, Wed 10 June, 17:00 to 19:00:

1. **The miss, for each hour:** the real count minus the guess. At 19:00 the real count
   was 3,037 and the guess 3,368, so the miss is −331.
2. **Drop the sign:** a miss of −331 counts as 331. Too high or too low, it is a miss
   of the same size.
3. **Average the sizes:** (101 + 335 + 331) ÷ 3 = 256 taxis.

| Hour, Wed 10 June | Real count | Guess: same hour last week, Wed 3 June | 1. Miss = real − guess | 2. Size of the miss |
|---|---|---|---|---|
| 17:00 | 3,348.5 | 3,247.5 | +101 | 101 |
| 18:00 | 3,387 | 3,052 | +335 | 335 |
| 19:00 | 3,037 | 3,368 | −331 | 331 |
| **3. Average of the sizes** | | | +35: the misses cancel | **256: the MAE** |

The 17:00 slot caught two readings each week, so it holds their average, with a half.

![MAE in three steps on Wed 10 June, 17:00 to 19:00: 1, the real count and the guess for each hour, misses +101, +335 and -331; 2, with their signs the misses cancel and average +35, which looks almost perfect; 3, without the sign their sizes average 256, the MAE; the empty 16:00 slot is left out](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_mae.png)

- **Why drop the sign.** Kept, the misses cancel: +101, +335 and −331 average +35,
  which looks almost perfect although every hour missed by 100 or more.
- **Empty slots are left out.** 16:00 that day is an empty slot, so it has no miss.
- **The MAE is in taxis**, like the data. A typical hour has about 2,375 free taxis, so
  an MAE of about 500 would mean the guess is off by about a fifth.

**Watch this before Step 1**: what the code in this task must do, on two real weeks of
`s` (it plays slowly, about a minute and a half).
![shift(168) on the hourly grid: week 1 slides one week later and becomes the guess for week 2; on Wed 10 June at 18:00 there were 3,387 taxis against a guess of 3,052; hours with no value or no guess are skipped; three misses, +101, +335 and -331, average +35 with their signs and 256 by size; the MAE is about 496 taxis](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_shift.gif)

**This time you write the prompt and send it.** Use the same four questions as in
Task 2. There is no example answer under them now: that is the point. The steps are
tagged with the loop: `PROMPT` → `GENERATE` → `VERIFY` → `REFINE`.

**Step 1.** `PROMPT` **Answer two questions** in the box *My decisions* below:

1. How many slots back is *the same hour last week*? (In `s` one slot is one hour, and
   a week has 7 days of 24 hours.)
2. Which hours can you compare, and which must you leave out? (The first week of `s` has
   no week before it, and 39 hours of `s` are empty.)

**My decisions:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 2.** `PROMPT` **Write your prompt, one line for each part**, in the box *My prompt*
below:

1. **What I have:** what would someone who has never seen your notebook need to know
   about the data? Ask yourself: what is it called, and is it a table or one column of
   numbers? How far apart are its values, and are some of them empty? What must stay
   unchanged?
2. **What I want:** what should the result be, and what is it called? Ask yourself: in
   plain words, what is the guess for one hour, say Wed 10 June at 18:00? Is the MAE one
   number, or one number per hour? The check cell after Step 5 needs two names: `snaive`
   for the guess, and `mae_snaive` for the MAE.
3. **How I will check:** what printed number would show it worked? Ask yourself: `s` has
   1,344 hours, so how many of them can have a guess to compare? What number would tell
   you the code used the right hours, and left out the rest?
4. **My decisions:** your two answers from Step 1. Ask yourself: could you add one worked
   hour from the table above, the way Task 2's prompt showed one finished box?

**My prompt:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 3.** `GENERATE` **Send it.** Start a new chat (**⋮** at the top right of the
Gemini panel, then **New chat**, then **Yes**), type your prompt and press Enter. When
the green cell appears, click **Accept and run** at the bottom of the chat panel, just
above the box where you type. If the code appears only in the chat, copy it, click
**+ Code**, paste and run it. If Gemini shows red and green lines inside the check cell
(first line `# TODO(you)`), click **Cancel**. Then scroll back down to this task.

**Step 4.** `VERIFY` **Run the check cell below.** You should see the MAE close to the
*Expected output* under the cell, and `[PASS] compare_to_baseline`. A NOTE here means
your MAE is wrong: the *Expected output* box says what each wrong number means.

**Step 5.** `REFINE` **Only if a number is wrong:** in the same chat, type one sentence
with your number and what the *Expected output* box says it means:

> **Prompt:** mae_snaive is [your number], but it should be about 496: [the reason
> from the box]. Fix only that.

Click **Accept and run**, then run the check cell again. If it is still
wrong after two tries, open our prompt below and compare it with yours.

<details><summary><b>Our prompt</b> (open it only after you have sent yours)</summary>

> **What I have:** `s`, an hourly pandas Series of taxi counts on a regular grid. Some
> hours are empty (NaN). Do not change `s`.
>
> **What I want:** `snaive`, which is `s` shifted by 168 slots (one week), and
> `mae_snaive`, the mean absolute error between `s` and `snaive`.
>
> **How I will check:** say how many hours were used. Explain in plain words what the
> code does.
>
> **My decisions:** compare only the hours where both `s` and `snaive` have a value:
> the first week has no week before it, and an empty hour has nothing to compare.

Where was yours different? A missing decision is the usual reason for a wrong MAE.
</details>

In [ ]:
# TODO(you): Accept and run Gemini's green cell first (wherever it landed) (it makes `snaive` and `mae_snaive`), then run this cell.
snaive = keep("snaive")
mae_snaive = keep("mae_snaive")

if filled(snaive, mae_snaive):
    print(f"seasonal-naive MAE: {mae_snaive:,.1f}")
    print(f"mean taxi count   : {s.mean():,.0f}")

# ---- verification contract — do not edit ----
# The baseline's MAE is compared with the series' standard deviation, which is
# roughly the error of always guessing the average. The report calls the first
# number "Model" because the check is general; here it is your baseline.
if filled(s):
    check.compare_to_baseline(mae_snaive, s.std(), metric="MAE")
check.report()

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** `seasonal-naive MAE: 495.6`
- **Line 2:** `mean taxi count   : 2,375`
- **Gemini's green cell** should say it used 1,114 hours: the hours where both `s` and `snaive` have a value.
- **The report:** `[PASS] compare_to_baseline`. It calls your guess "Model" and compares its MAE with about 720, the spread of `s` (its standard deviation): roughly the miss of always guessing the average.
- **If your MAE is about 824:** the empty values were filled with 0 taxis: the 39 empty hours, and the first week, which has no week before it to copy. **If it is about 411:** hours with no guess were counted as a miss of 0. Use only the hours where both exist.

**Run the cell below.** **Look for** how close "the same hour last week" tracks the
real line, and where it misses.

In [ ]:
if filled(snaive):
    show.compare(s.iloc[-24 * 14:], same_hour_last_week=snaive.iloc[-24 * 14:],
                 title="Last two weeks: actual against the seasonal-naive guess")

**What you should see:** the dashed guess follows the real line closely most of
the time, because the daily cycle repeats. Where they part is where something
unusual happened in one of the two weeks. Those misses are what the MAE averages:
about 496 taxis an hour, against a mean of about 2,375. A better model would have to
explain them.

## Task 5 — See the daily and weekly cycles (10 minutes)

![Where you are: step 5 of 5, see the cycles](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab03_plan_step5.png)

*Topic 3: seasonality, a pattern that repeats every day or every week.*

**What the cycles are for.** They show when taxis are scarce, which is when to send
more out. They are why Task 4's guess works: the day and the week repeat. And they give
the numbers to count back: 24 slots for a day, 168 for a week.

1. **Run the cell below.** It draws two line charts (dots joined by a line): the
   average by hour of day (24 dots), and by day of the week (7 dots).
2. **Look for** the lowest and highest dots in each chart.

In [ ]:
if filled(s):
    show.profile(s, by="hour", title="Mean taxis available by hour of day")
    show.profile(s, by="dow", title="Mean taxis available by day of week")

**What you should see:** in the first chart, a deep dip in the small hours of the
night and a high, uneven stretch from late morning to late evening. In the second,
two days sit well below the other five. Read the hour of the lowest dot off the
first chart's bottom axis.

3. **Write** the hour at which availability is lowest in the box under the charts.
4. `PROMPT` `GENERATE` `VERIFY` **Optional:** start a new chat (**⋮** at the top right of the Gemini panel, then **New chat**, then **Yes**) and type this prompt:

   > **Prompt:** Draw a bar chart of the mean of `s` by hour of day.

   Accept and run its cell. Its lowest bar should sit at the same hour as
   the lowest dot in the first chart above.

**Hour of lowest availability:**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

About 3 am. The `03:00` slot has the lowest average, about 1,785 taxis, with 4 am
and 5 am only a little higher (about 1,890 and 1,840). Most readings arrive just before
the hour, so the 03:00 slot mostly holds readings taken at about 03:59: the quietest
time is just before 4 am. Either answer is fine if you say why. The busiest end is the
evening, about 2,780 at 8 pm. A good answer gives the hour, not a day. The common
slip is reading the second chart, the day-of-week one, and writing "Saturday": that
is the lowest day (about 1,880, with Sunday about 1,940), not the lowest hour.
Another is writing midnight, because the line starts there; at about 2,280 it is far
from the bottom.
</details>

---
## Predict: How many taxis at 8 am on Monday? (15 minutes)

*The Prediction League: one question per lab, settled by reality. Individual work: you enter on your own, under your own name.*

**The question:** Taxis available across Singapore in the 8 o'clock reading on the Monday of the target week. The target is the week of Monday 2026-11-09.

**How it works:** you enter a short function, `predict(history)`, not a number. The league page runs it on the newest data, and the number it gives when you click
**Submit** is your entry: it does not change after you submit. When the target
is over, the real value is compared with yours. League page (with the closing time):
https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=3

**Step 1.** **Run the cell below.** It prints the number from the simple rule,
*the average of every Monday 8 o'clock reading so far*, on the lab's own file. The league page uses newer data,
so its number will differ.

In [ ]:
fetch("sg_taxi_hourly.csv")
taxi = pd.read_csv("sg_taxi_hourly.csv", parse_dates=["timestamp"])

def predict(history):
    # the baseline: the average of every Monday 8 o'clock reading so far
    ts = history["timestamp"]
    mondays = history[(ts.dt.weekday == 0) & (ts.dt.hour == 8)]
    print("Mondays at 8:", mondays["taxi_count"].tolist())
    return float(mondays["taxi_count"].mean())

print("The baseline predicts", round(predict(taxi), 1), "taxis")

**What you should see:** *The baseline predicts 2399.4 taxis*.

**Step 2.** **Write the prompt in four parts, then send it.** The same four parts as Lab 3,
because they answer the four things Gemini cannot know for you:

| Part | The question it answers | Here |
|---|---|---|
| What I have | What is the data, and what does it not hold? | `history`, its columns, and the dates it covers |
| What I want | What should come back, and what is it called? | a function `predict(history)` that returns one number |
| How I will check | What printed line would show it worked? | the data span printed; one sentence on what was dropped |
| My decisions | Where could two people do this differently? | **your idea**, in plain words |

**Why four parts?** Gemini cannot know what data you have, what shape you want back,
how you will check it, or what you decided. Parts 1 to 3 are facts about the contest, so they
are written for you below. Part 4 is yours: replace *MY IDEA* with your idea in one or
two plain sentences (for example "use only the last four Mondays", or "last Monday's reading,
nudged by this week's trend"). If you have no idea yet, write "the simple rule" and improve
it after Step 4.

Start a new chat (**⋮** › **New chat** › **Yes**), paste the prompt with your part 4 filled in, and press Enter.

> **What I have:** a pandas table `history` with the columns timestamp (one reading per hour, Singapore time) and taxi_count, oldest first. It holds one row per hour from 29 May 2026 up to the day before entries close: nothing earlier, no other columns, no forecast, no internet.
> **What I want:** a function predict(history) that returns one number with float(...): taxis available across Singapore in the 8 o'clock reading on the Monday of the target week. Only the function and the import lines it needs, in a new cell; do not change the cells above.
> **How I will check:** the first lines of the function print the first date, the last date and the row count. No internet calls and no try/except that hides an error. After the code, one sentence on the idea and one naming anything you dropped from my idea and why.
> **My decisions:** MY IDEA. Check it against what history holds before you write: if it needs something that is not there, say so, then shrink the idea or fall back to the simple rule (the average of every Monday 8 o'clock reading so far). Never write a branch that can only run on data that is not there.

**My idea (part 4), in plain words:**

Click **Accept and run** on Gemini's green cell. It prints nothing; it only defines the
function. If Gemini shows red and green lines inside the Step 1 cell, click **Cancel** and type *put it in a new cell*.

**Step 3.** **Run the cell below.**

In [ ]:
# TODO(you): run Gemini's green cell first (Step 2)
show_predict(taxi, "taxis")

**What you should see:** *Your predict says* and a number, usually between 1,500 and 4,000, then your
code between two dashed lines. If it says *Gemini's predict has not run yet*, do what that message says.
If you see a red error, paste the last red line into Gemini with *fix this*, click
**Accept and run**, and run this cell again.

<details><summary><b>Using ChatGPT, Claude or another AI instead of Gemini?</b> Open this for a prompt to copy</summary>

That AI cannot see this notebook, so it needs a longer prompt that spells out every
rule the league checks. Use it in place of Steps 2 and 3:

1. **Copy the prompt** in the grey box below.
2. **Paste it** into the other AI's chat box. Do not send it yet.
3. **Write your idea.** In the line *My idea: WRITE YOUR IDEA HERE*, replace *WRITE YOUR IDEA HERE* with your idea, in plain words.
   Plain text is fine: no bold or italics needed. If you have no idea yet, delete the whole line: the AI then chooses one.
4. **Optional:** to let the AI look at the numbers first, download the data,
   https://stanleysp-eng.github.io/EP0418-labs/data/taxi_hourly.csv (the same file your
   function receives, updated every night), and attach it to the chat.
5. **Send it.** Copy the code the AI gives back, then do **Step 4** below: paste that
   code into the league page's **Code** box, in place of the code from Step 3.

```text
I am entering a prediction contest. Write one Python function for it.

The question: Taxis available across Singapore in the 8 o'clock reading on the Monday of the target week. The target is the week of Monday 2026-11-09 to Sunday 2026-11-15.
The simple rule to beat: the average of every Monday 8 o'clock reading so far.
My idea: WRITE YOUR IDEA HERE

First, check my idea against the data before you write anything. history holds one row per hour from 29 May 2026 up to the day before I submit. There is nothing earlier, no other columns, and no weather forecast or internet. If my idea needs something history does not have (past years, a season from another year, wind, temperature, a live feed), say so plainly, then either shrink the idea to what the data can support or fall back to the simple rule. Never write a branch that can only run on data that is not there.

The contest runs the function automatically, so follow these rules exactly:
1. Name it predict, with exactly one input called history:  def predict(history):
2. history is a pandas DataFrame that the contest passes in, oldest row first. Its column names are exactly: timestamp, taxi_count. What they hold: timestamp (one reading per hour, Singapore time) and taxi_count. The timestamp column is timezone-aware (Asia/Singapore): compare it only with timezone-aware times, for example pd.Timestamp("2026-10-26", tz="Asia/Singapore").
3. Do not read any file, do not call the internet and do not ask for input. All the data is already in history. If I attach a data file, look at it only to choose the idea; the function must still take everything from history.
4. The first lines of the function print the first date, the last date and the number of rows, so I can see what the code was given.
5. No try/except that hides a problem. If something fails, let it fail so I can see it.
6. Return one number with return float(...). Not a list, a table, text, None or NaN.
7. Use only pandas, numpy and the Python standard library, and finish in under 30 seconds.
8. Give only the import lines and the function: no example data, no test code, nothing that calls predict.

After the code: one sentence on what the code really does with this data, and one sentence naming any part of my idea you dropped and why. If you dropped nothing, say so.
```
</details>

**Step 4. Submit on the league page.**
1. **Copy your code** from the Step 3 output: everything between the two dashed lines.
2. **Open** https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=3 in a new tab and type your name as on the class list (the same name every week).
3. **Paste** it into the **Code** box: click in it, Ctrl+A, Ctrl+V (Cmd on a Mac).
4. **Click Run tests** (about 15 seconds the first time). **What you should see:** *All 4 cases ran* and four tabs: three past cases and *Your entry*. On *Your entry*, Actual says *not known yet*, and the number differs from Colab's because the page uses newer data: the page's number is the one that counts. A red *missed by* only means the simple rule did better on that case. A **red heading** instead of *All 4 cases ran* is an error: copy the red text under the case marked red (or under the heading), paste it into the same Gemini chat with *fix this*, click **Accept and run**, run Step 3 again, then paste and **Run tests** again.
5. **Click Submit.** *Submitted at ...* and *Your entry is ...* confirm it. Submit again any time before the close to replace it; the latest one counts.

**Write it down** (on the sheet, or double-click here): your entry (the number the league page gave at Submit),
the baseline (Step 1), and the one-sentence idea Gemini gave. After the target, the
real answer is on the league page.

**My number / the baseline / the idea:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

There is no single right answer here. A good entry gives all three parts: your
function's number, the baseline (2399.4 taxis), and an idea that uses the data. The Monday 8 o'clock readings in the file are 3,841, 2,065, 2,460, 2,249, 2,732, 2,040, 2,006 and 1,802. The first is far above the rest: without it the average is about 2,193 instead of 2,399, and the median of all eight is 2,157. Ideas that use this: the median, or the average of the last few Mondays.
The common weak answer is "Gemini wrote it". It says who wrote the code, not what the idea is. The past-case
tabs on the league page show whether the idea would have beaten the simple rule
lately; three cases are too few to prove it.
</details>

---
## Read before you run

An assistant was asked to put the taxi feed onto a regular hourly grid and “make sure there are no missing values”. It wrote the code below.

The code is switched off: every line starts with `#`, so it cannot run yet.

1. **Read it, and answer Question 1** in the box under the code cell, or on the
   sheet. Do not run it yet.
2. **Then run it.** Delete the `# ` at the start of each line below the first line,
   run the cell, and compare what it prints with your answer.
3. **Answer Question 2.**

**Question 1.** A taxi company will use `their_s` to find the hours when free taxis are scarcest, and send more cars out then. What does `fillna(0)` write into the 39 empty hours, such as 04:00 on 30 May? Which hours would the company now think are the scarcest, and would it be right?

**Question 2.** What one check would have caught this before anyone used the result? Say what you would print or look at, and what it would show.

<details><summary>Hint 1: where to look</summary>

What value does `fillna(0)` write into an empty hour? Is zero a measurement of taxis, or the absence of one?

</details>
<details><summary>Hint 2: closer to the answer</summary>

The two printed lines show it. The 39 empty hours become hours with 0 taxis: the minimum falls to 0 and the mean drops, and the `regular_cadence` check from Task 2 would still pass.

</details>



**Optional:** paste the code into a new Gemini chat with the prompt *Explain this code
in plain words.* Does its explanation name the fault? Often it does not: an
explanation that reads well is not evidence the code is right. To fix the code, type
*Fix the problem in this code that Hint 2 describes. Say in one sentence what you
changed.* If that one change matches Hint 2, click **Accept and run**; if it changed
other things too, click **Cancel** and type *fix only that, nothing else*. You should
see: the *after* line no longer shows min 0; it matches the *before* line (min 24, mean about 2,375).

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# their_s = raw.set_index("timestamp").taxi_count.resample("h").mean()
# print("before: min", their_s.min(), " mean", round(their_s.mean()))
# their_s = their_s.fillna(0)
# print("after:  min", their_s.min(), " mean", round(their_s.mean()))

**Question 1, my answer:**

**Question 2, my answer:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Quiz yourself before you leave (5 minutes)

The checks tested the code. This quiz tests what you understood. In Gemini, start a new chat
(**⋮** › **New chat** › **Yes**), paste this into the chat box and press Enter:

> **Rules:** Ask one short question at a time, in plain words. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer.
>
> **Task:** Quiz me on this lab: five questions on these ideas: why the taxi timestamps are not on the hour; what resampling onto a regular grid does to the gaps; why shift(168) only means one week on a regular grid; what a seasonal-naive forecast is; what mean absolute error measures. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks one question at a time in the chat panel; answer in the chat box, in
your own words, without looking back. Ignore the suggested replies Gemini shows. If you
use ChatGPT, Claude or Copilot instead, first delete the sentence *Take the examples from
the code and outputs in this notebook.*: they cannot see your notebook.

Then write your score and the idea to revise below. Your tutor asks for them.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Questions to close the lab

**Write two answers** in the box below, or on the last page of the sheet. Each one
asks you to use what you saw today, not to repeat a definition.

1. In this lab's file, counting 168 rows back always landed within half an hour of one
   week back. Why? What would have to happen to the feed for 168 rows back to land on
   the wrong hour, and why does the hourly grid not have that problem?
2. A taxi operator plans, each evening, how many taxis to send out in each hour of the
   coming night. Is an hourly grid fine enough for that? What would a daily grid, one
   average per day, hide? Use what Task 5 showed about the night.

**My two answers:**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

1. The feed never missed a reading, so 168 rows were always about 168 hours: the
   readings drift by seconds, and 168 rows back landed between 6 days 23 h 33 min and
   7 days 0 h 10 min back. If readings go missing, the rows close up and every count
   after the gap goes back too far: 3 lost readings and 168 rows back lands 3 hours
   early, with no error. On the grid a lost reading leaves an empty slot, so 168 slots
   back is always exactly one week.
2. Hourly is fine: the operator decides hour by hour, and the hourly grid shows the dip
   at about 3 to 4 am, when free taxis fall to about 1,785. A daily grid gives one
   number per day, about 2,375, and hides the dip completely: the operator could not
   tell 3 am from 8 pm. A good answer names the decision's own step (each hour) and
   what the coarser grid averages away (the night).
</details>

---
## Before you leave

**One thing to notice:** the smallest reading in this file is 24 taxis across all of
Singapore, at 21:54 on 17 July, when the average is about 2,375. That is not
plausible: it is a real glitch in the feed, not something we added. Lab 4 is about
what to do with values like that.

**Before you leave**, fill in the *Mistake caught* box below (double-click it), or
write it on the last page of the Lab 3 sheet. If Gemini's first green cell for Task 2 gave
outcome B or C, write about that. If it gave A, write about the read-before-you-run
code, for example *I expected empty hours to stay empty · fillna(0) made them 0 taxis ·
the minimum fell from 24 to 0.*

**Mistake caught:** *(three short lines: 1. what I expected, 2. what was there instead, 3. how I found it. If Gemini gave outcome A first time, use the read-before-you-run code above instead.)*

*(Your answer box: double-click this cell. Type your answer here.)*